In [2]:
import pandas as pd
from pathlib import Path

In [15]:
# GTFS raw files location
DATA_DIR = Path("data")
GTFS_DIR = DATA_DIR / "raw" / "ttc_gtfs"
OUTPUT_DIR = Path("output")

routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")

# 50 selected TTC bus route short_names
SELECTED_ROUTES = ["7", "11", "12", "24", "25", "29", "32", "34", "35", "36", "37", "38", "39", "40", "41", "43", "45", "47", "52", "53",
    "54", "59", "60", "63", "68", "72", "73", "79", "84", "85", "86", "89", "94", "95", "96", "97", "100", "102", "108",
    "110", "116", "123", "129", "133", "165", "924", "925", "929", "939", "960"]

C:\Users\amras\AppData\Local\Temp\ipykernel_34500\1741196784.py:7: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(GTFS_DIR / "trips.txt")
C:\Users\amras\AppData\Local\Temp\ipykernel_34500\1741196784.py:8: DtypeWarning: Columns (0,5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")


In [18]:
def load_gtfs():
    # Keep only selected routes
    routes_filtered = routes[routes["route_short_name"].astype(str).isin(SELECTED_ROUTES)].copy()

    if routes_filtered.empty:
        raise ValueError(
            "No selected routes found in routes.txt."
        )

    # Trips for selected routes
    trips_filtered = trips[trips["route_id"].isin(routes_filtered["route_id"])].copy()
    # Stop times for selected trips
    stop_times_filtered = stop_times[stop_times["trip_id"].isin(trips_filtered["trip_id"])].copy()

    # First stop of each trip
    first_stop_per_trip = (stop_times_filtered.sort_values("stop_sequence").groupby("trip_id", as_index=False)
        .first()[["trip_id", "arrival_time"]])

    # Merge trip information
    trip_details = (trips_filtered.merge(
            routes_filtered[["route_id", "route_short_name", "route_long_name", "route_color", "route_text_color"]],
            on="route_id", how="left"
        ).merge(
            first_stop_per_trip,
            on="trip_id", how="left"
        )
    ) 

    # Extract scheduled hour
    trip_details["scheduled_hour"] = (trip_details["arrival_time"].str.split(":").str[0].astype(int).mod(24))
    # Extract compass direction from trip_headsign
    def extract_direction(text):
        if not isinstance(text, str):
            return "UNKNOWN"

        text = text.upper().strip()
        if text.startswith("EAST"):
            return "E"
        elif text.startswith("WEST"):
            return "W"
        elif text.startswith("NORTH"):
            return "N"
        elif text.startswith("SOUTH"):
            return "S"
        else:
            return "UNKNOWN"

    trip_details["direction"] = (trip_details["trip_headsign"].apply(extract_direction))
    
   # One hot encode direction
    direction_dummies = pd.get_dummies(trip_details["direction"], prefix="dir", dtype=int)
    # Ensure model columns exist
    direction_columns = ["dir_E","dir_N","dir_S","dir_W","dir_UNKNOWN"]

    for col in direction_columns:
        if col not in direction_dummies.columns:
            direction_dummies[col] = 0

    trip_details = pd.concat([trip_details, direction_dummies[direction_columns]], axis=1)

    # Final reference dataset
    trip_details = trip_details[
        ["trip_id","route_id","route_short_name","route_long_name","route_color","route_text_color","direction_id","scheduled_hour",
         "dir_E","dir_N","dir_S","dir_W","dir_UNKNOWN"]].copy()

    return trip_details

# Create and save reference file
trip_details = load_gtfs()

trip_details.to_csv(OUTPUT_DIR / "TTC_trip_details.csv", index=False)
print(f"Saved {len(trip_details):,} trips")

Saved 50,643 trips


In [19]:
# CREATE STOP REFERENCE FOR SELECTED ROUTES
def create_stops_reference():
    # Filter selected routes
    routes_filtered = routes[routes["route_short_name", "route_id"].astype(str).isin(SELECTED_ROUTES)]

    # Get trips for selected routes
    trips_filtered = trips[trips["route_id"].isin(routes_filtered["route_id"])]

    # Get stops used by those trips
    stop_ids = stop_times[stop_times["trip_id"].isin(trips_filtered["trip_id"])]["stop_id"].unique()

    # Get stop information
    stops_reference = stops[stops["stop_id"].isin(stop_ids)].copy()
    # Keep useful columns
    stops_reference = stops_reference[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

    # Remove duplicates
    stops_reference = (stops_reference.drop_duplicates("stop_id").reset_index(drop=True))

    return stops_reference

stops_reference = create_stops_reference()
stops_reference.to_csv( OUTPUT_DIR / "stops_reference.csv", index=False)
print(f"Saved {len(stops_reference):,} stops")

KeyError: ('route_short_name', 'route_id')

In [13]:
# CREATE ROUTE REFERENCE
def create_route_shapes():
    # Filter selected routes
    routes_filtered = routes[routes["route_short_name"].astype(str).isin(SELECTED_ROUTES)].copy()
    # Trips belonging to selected routes
    trips_filtered = trips[trips["route_id"].isin(routes_filtered["route_id"])].copy()

    # Keep shape IDs used by selected routes
    shape_ids = trips_filtered["shape_id"].unique()
    shapes_filtered = shapes[shapes["shape_id"].isin(shape_ids)].copy()
    
    # Add route information
    route_shapes = (trips_filtered[["route_id","direction_id","shape_id"]].drop_duplicates()
        .merge(routes_filtered[["route_id","route_short_name","route_long_name","route_color","route_text_color"]],
            on="route_id",how="left")
        .merge(shapes_filtered,on="shape_id",how="left"))

    # Sort for drawing lines
    route_shapes = route_shapes.sort_values(["route_short_name","direction_id","shape_id","shape_pt_sequence"])

    return route_shapes

route_shapes = create_route_shapes()
route_shapes.to_csv(OUTPUT_DIR / "routes_reference.csv", index=False)
print(f"Saved {len(route_shapes):,} shape points")

Saved 337,224 shape points
